In [2]:
import os
import configparser
from pyspark.sql import SparkSession

POLARIS_PRINCIPAL = "dev"  # "dev", "analyst" or "silver_analyst"


PROJECT_DIR = "/home/onyxia/work/bigdataframework-lab3"

# Identifiants S3 : profil "default" (celui qui fonctionne)
cfg = configparser.ConfigParser()
cfg.read(os.path.expanduser("~/.aws/credentials"))
ACCESS_KEY = cfg["default"]["aws_access_key_id"]
SECRET_KEY = cfg["default"]["aws_secret_access_key"]
SESSION_TOKEN = cfg["default"].get("aws_session_token", "")

# Identifiants Polaris : fichier .polaris-credentials.env du projet
polaris_creds = {}
with open(f"{PROJECT_DIR}/.polaris-credentials.env") as f:
    for line in f:
        line = line.strip()
        if "=" in line:
            key, value = line.split("=", 1)
            polaris_creds[key] = value.strip('"')
POLARIS_CREDENTIAL = polaris_creds[f"POLARIS_{POLARIS_PRINCIPAL.upper()}_CREDENTIAL"]

BUCKET = os.environ["KUBERNETES_NAMESPACE"]
POLARIS_URI = "http://polaris:8181"
POLARIS_CATALOG = BUCKET
ICEBERG_PACKAGE = "org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.11.0"

spark = (
    SparkSession.builder
    .appName("lab-polaris")
    .master("local[*]")
    # S3 access
    .config("spark.hadoop.fs.s3a.endpoint", f"https://{os.environ['AWS_S3_ENDPOINT']}")
    .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY)
    .config("spark.hadoop.fs.s3a.session.token", SESSION_TOKEN)
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider",
    )
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.jars.packages", ICEBERG_PACKAGE)
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    # Hadoop catalog of the Spark lab
    .config("spark.sql.catalog.lab", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lab.type", "hadoop")
    .config("spark.sql.catalog.lab.warehouse", f"s3a://{BUCKET}/warehouse")
    # Polaris catalog
    .config("spark.sql.catalog.polaris", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.polaris.type", "rest")
    .config("spark.sql.catalog.polaris.uri", f"{POLARIS_URI}/api/catalog")
    .config("spark.sql.catalog.polaris.warehouse", POLARIS_CATALOG)
    .config("spark.sql.catalog.polaris.credential", POLARIS_CREDENTIAL)
    .config("spark.sql.catalog.polaris.scope", "PRINCIPAL_ROLE:ALL")
    .config("spark.sql.catalog.polaris.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Polaris principal:", POLARIS_PRINCIPAL)
spark

:: loading settings :: url = jar:file:/usr/local/lib/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/onyxia/.ivy2.5.2/cache
The jars for the packages stored in: /home/onyxia/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e5a866df-b364-45a0-a3f7-3a9b2a9c1bed;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.11.0 in central
:: resolution report :: resolve 547ms :: artifacts dl 6ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.11.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   | 

Polaris principal: dev


In [5]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lab.silver")

orders_parquet = spark.read.parquet(f"s3a://{BUCKET}/silver/spark/orders.parquet")
orders_parquet.writeTo("lab.silver.orders").createOrReplace()

users_parquet = spark.read.parquet(f"s3a://{BUCKET}/silver/spark/users.parquet")
users_parquet.writeTo("lab.silver.users").createOrReplace()

print("orders :", spark.table("lab.silver.orders").count())
print("users  :", spark.table("lab.silver.users").count())
spark.sql("SHOW NAMESPACES IN lab").show()

orders : 2830
users  : 50
+---------+
|namespace|
+---------+
|   silver|
+---------+



In [6]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.silver")

spark.sql(f"""
    CALL polaris.system.register_table(
      table => 'silver.orders',
      metadata_file => 's3a://{BUCKET}/warehouse/silver/orders/metadata/v1.metadata.json'
    )
""").show()

spark.sql("SELECT count(*) AS orders FROM polaris.silver.orders").show()

+-------------------+-------------------+----------------------+
|current_snapshot_id|total_records_count|total_data_files_count|
+-------------------+-------------------+----------------------+
|1195310225212103902|               2830|                     2|
+-------------------+-------------------+----------------------+

+------+
|orders|
+------+
|  2830|
+------+



In [7]:
spark.sql("SHOW TABLES IN polaris.silver").show()
spark.sql("SELECT snapshot_id, operation, committed_at FROM polaris.silver.orders.snapshots").show(truncate=False)

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|   silver|   orders|      false|
+---------+---------+-----------+

+-------------------+---------+-----------------------+
|snapshot_id        |operation|committed_at           |
+-------------------+---------+-----------------------+
|1195310225212103902|overwrite|2026-11-06 18:04:13.687|
+-------------------+---------+-----------------------+



In [8]:
stg_users = spark.table("lab.silver.users")
stg_users.writeTo("polaris.silver.users").createOrReplace()

spark.sql("SELECT count(*) AS users FROM polaris.silver.users").show()

+-----+
|users|
+-----+
|   50|
+-----+



In [9]:
spark.sql("ALTER TABLE polaris.silver.users ADD COLUMN country STRING")
spark.sql("DESCRIBE polaris.silver.users").show()

+---------+---------+-------+
| col_name|data_type|comment|
+---------+---------+-------+
|  user_id|   string|   NULL|
| username|   string|   NULL|
|     name|   string|   NULL|
|      sex|   string|   NULL|
|    email|   string|   NULL|
|birthdate|     date|   NULL|
|   street|   string|   NULL|
|     city|   string|   NULL|
|    state|   string|   NULL|
| zip_code|   string|   NULL|
|  country|   string|   NULL|
+---------+---------+-------+



In [2]:
spark.sql("SHOW TABLES IN polaris.silver").show()
spark.sql("SELECT product, count(*) FROM polaris.silver.orders GROUP BY product").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|   silver|    users|      false|
|   silver|   orders|      false|
+---------+---------+-----------+



+---------+--------+
|  product|count(1)|
+---------+--------+
|    donut|     462|
|   cookie|     467|
|croissant|     491|
|    bread|     458|
|  brioche|     468|
|    drink|     484|
+---------+--------+



In [3]:
tests = {
    "INSERT": "INSERT INTO polaris.silver.orders SELECT * FROM polaris.silver.orders LIMIT 1",
    "ALTER":  "ALTER TABLE polaris.silver.users ADD COLUMN phone STRING",
}

for nom, requete in tests.items():
    print(f"===== {nom} =====")
    try:
        spark.sql(requete)
        print("⚠️ Réussi (anormal pour analyst)")
    except Exception as e:
        for ligne in str(e).splitlines():
            if "Forbidden" in ligne or "not authorized" in ligne.lower() or "Exception:" in ligne:
                print(ligne[:300])
                break

===== INSERT =====


18:08:20.382 [Thread-4] ERROR org.apache.spark.sql.execution.datasources.v2.AppendDataExec - Data source write support IcebergBatchWrite(table=polaris.silver.orders, format=PARQUET) is aborting.
18:08:20.459 [Thread-4] ERROR org.apache.spark.sql.execution.datasources.v2.AppendDataExec - Data source write support IcebergBatchWrite(table=polaris.silver.orders, format=PARQUET) aborted.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'analyst' with activated PrincipalRoles '[data_analyst]' and activated grants via '[data_analyst, catalog_reader]' is not authorized for op ADD_TABLE_SNAPSHOT
===== ALTER =====
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'analyst' with activated PrincipalRoles '[data_analyst]' and activated grants via '[data_analyst, catalog_reader]' is not authorized for op ADD_TABLE_SCHEMA


In [2]:
spark.sql("""
    SELECT snapshot_id, operation, committed_at
    FROM polaris.silver.orders.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-------------------+---------+-----------------------+
|snapshot_id        |operation|committed_at           |
+-------------------+---------+-----------------------+
|1195310225212103902|overwrite|2026-11-06 18:04:13.687|
+-------------------+---------+-----------------------+



In [3]:
print("=== Snapshots AVANT ===")
spark.sql("SELECT snapshot_id, operation, committed_at FROM polaris.silver.users.snapshots ORDER BY committed_at").show(truncate=False)

print("=== Fichiers de métadonnées AVANT ===")
spark.sql("SELECT timestamp, file FROM polaris.silver.users.metadata_log_entries ORDER BY timestamp").show(truncate=False)

=== Snapshots AVANT ===
+-------------------+---------+-----------------------+
|snapshot_id        |operation|committed_at           |
+-------------------+---------+-----------------------+
|6231320906659012603|overwrite|2026-11-06 18:05:44.216|
+-------------------+---------+-----------------------+

=== Fichiers de métadonnées AVANT ===
+-----------------------+-------------------------------------------------------------------------------------------------------------------+
|timestamp              |file                                                                                                               |
+-----------------------+-------------------------------------------------------------------------------------------------------------------+
|2026-11-06 18:05:45.132|s3://user-m-lebourdais-ece/warehouse/silver/users/metadata/00000-4b7c510e-d4d2-4de9-a253-9890e2942548.metadata.json|
|2026-11-06 18:06:24.575|s3://user-m-lebourdais-ece/warehouse/silver/users/metadata/00001

In [4]:
spark.sql("ALTER TABLE polaris.silver.users DROP COLUMN country")
spark.sql("DESCRIBE polaris.silver.users").show()

print("=== Snapshots APRÈS ===")
spark.sql("SELECT snapshot_id, operation, committed_at FROM polaris.silver.users.snapshots ORDER BY committed_at").show(truncate=False)

print("=== Fichiers de métadonnées APRÈS ===")
spark.sql("SELECT timestamp, file FROM polaris.silver.users.metadata_log_entries ORDER BY timestamp").show(truncate=False)

+---------+---------+-------+
| col_name|data_type|comment|
+---------+---------+-------+
|  user_id|   string|   NULL|
| username|   string|   NULL|
|     name|   string|   NULL|
|      sex|   string|   NULL|
|    email|   string|   NULL|
|birthdate|     date|   NULL|
|   street|   string|   NULL|
|     city|   string|   NULL|
|    state|   string|   NULL|
| zip_code|   string|   NULL|
+---------+---------+-------+

=== Snapshots APRÈS ===
+-------------------+---------+-----------------------+
|snapshot_id        |operation|committed_at           |
+-------------------+---------+-----------------------+
|6231320906659012603|overwrite|2026-11-06 18:05:44.216|
+-------------------+---------+-----------------------+

=== Fichiers de métadonnées APRÈS ===
+-----------------------+-------------------------------------------------------------------------------------------------------------------+
|timestamp              |file                                                                 

In [5]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.gold")

(
    spark.table("polaris.silver.orders")
    .groupBy("product")
    .count()
    .writeTo("polaris.gold.orders_by_product")
    .createOrReplace()
)

spark.sql("SHOW NAMESPACES IN polaris").show()
spark.sql("SELECT * FROM polaris.gold.orders_by_product").show()

+---------+
|namespace|
+---------+
|   silver|
|     gold|
+---------+

+---------+-----+
|  product|count|
+---------+-----+
|    donut|  462|
|   cookie|  467|
|croissant|  491|
|    bread|  458|
|  brioche|  468|
|    drink|  484|
+---------+-----+



In [6]:
POLARIS_PRINCIPAL = "silver_analyst"  # "dev", "analyst" or "silver_analyst"

In [ ]:
tests = {
    "Lire silver.orders": "SELECT count(*) AS n FROM polaris.silver.orders",
    "Lister silver":      "SHOW TABLES IN polaris.silver",
    "Lire gold":          "SELECT * FROM polaris.gold.orders_by_product",
    "Lister gold":        "SHOW TABLES IN polaris.gold",
}

for nom, requete in tests.items():
    print(f"===== {nom} =====")
    try:
        spark.sql(requete).show()
        print("autorise")
    except Exception as e:
        ligne = next((l for l in str(e).splitlines() if "Forbidden" in l or "not authorized" in l.lower() or "Exception:" in l), str(e).splitlines()[0])
        print("refus :", ligne[:250])

===== Lire silver.orders =====
+----+
|   n|
+----+
|2830|
+----+

✅ Autorisé
===== Lister silver =====
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|   silver|    users|      false|
|   silver|   orders|      false|
+---------+---------+-----------+

✅ Autorisé
===== Lire gold =====
+---------+-----+
|  product|count|
+---------+-----+
|    donut|  462|
|   cookie|  467|
|croissant|  491|
|    bread|  458|
|  brioche|  468|
|    drink|  484|
+---------+-----+

✅ Autorisé
===== Lister gold =====
+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|     gold|orders_by_product|      false|
+---------+-----------------+-----------+

✅ Autorisé


In [3]:
spark.sql("DROP TABLE IF EXISTS polaris.silver.users")
spark.sql("DROP TABLE IF EXISTS polaris.gold.orders_by_product")
spark.sql("DROP NAMESPACE IF EXISTS polaris.gold")

spark.sql("SHOW NAMESPACES IN polaris").show()
spark.sql("SHOW TABLES IN polaris.silver").show()

+---------+
|namespace|
+---------+
|   silver|
+---------+

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|   silver|   orders|      false|
+---------+---------+-----------+



In [4]:
spark.stop()